# Generating ONNX Training graph

Generating artifacts based on forward-only graph and the specified loss function using onnxblock library

In [1]:
from pathlib import Path

import onnx
import onnxruntime.training.onnxblock as onnxblock

In [2]:
# Creating a class with a Loss function
class CustomTrainingBlock(onnxblock.TrainingBlock):
    def __init__(self):
        super(CustomTrainingBlock, self).__init__()
        self.loss = onnxblock.loss.CrossEntropyLoss()

    def build(self, output_name):
        return self.loss(output_name), output_name

In [3]:
model_name = 'mobilenetv2_100'
# artifacts_path = Path(f'../artifacts/{model_name}/frozen_layer_21_classes_10/hailo8_optim_3_comp_0')
# onnx_artifacts_path = artifacts_path / f'{model_name}_quantized_model_hailo.onnx'

artifacts_path = Path(f'../artifacts/{model_name}/frozen_layer_21_classes_10/cpu')
artifacts_path.mkdir(parents=True, exist_ok=True)
onnx_artifacts_path = artifacts_path.parent / f'{model_name}.onnx'

onnx_model = onnx.load_model(onnx_artifacts_path)

# for param in onnx_model.graph.initializer:
#     print(param.name)

In [4]:
training_block = CustomTrainingBlock()
for param in onnx_model.graph.initializer:
    if 'frozen_features' in param.name or 'bn' in param.name or 'downsample.1' in param.name:
        print(param.name.ljust(40), '\t--->\tfrozen')
        training_block.requires_grad(param.name, False)
    else:
        print(param.name.ljust(40), '\t--->\tlearnable')
        training_block.requires_grad(param.name, True)

# Building training graph and eval graph
model_params = None
with onnxblock.base(onnx_model):
    build_output = training_block(*[output.name for output in onnx_model.graph.output])
    print('Graph output nodes:', build_output)
    training_model, eval_model = training_block.to_model_proto()
    model_params = training_block.parameters()

# Building the optimizer graph
optimizer_block = onnxblock.optim.AdamW()
with onnxblock.empty_base() as accessor:
    _ = optimizer_block(model_params)
    optimizer_model = optimizer_block.to_model_proto()

frozen_features.0.weight                 	--->	frozen
frozen_features.1.weight                 	--->	frozen
frozen_features.1.bias                   	--->	frozen
frozen_features.1.running_mean           	--->	frozen
frozen_features.1.running_var            	--->	frozen
frozen_features.2.conv_dw.weight         	--->	frozen
frozen_features.2.bn1.weight             	--->	frozen
frozen_features.2.bn1.bias               	--->	frozen
frozen_features.2.bn1.running_mean       	--->	frozen
frozen_features.2.bn1.running_var        	--->	frozen
frozen_features.2.conv_pw.weight         	--->	frozen
frozen_features.2.bn2.weight             	--->	frozen
frozen_features.2.bn2.bias               	--->	frozen
frozen_features.2.bn2.running_mean       	--->	frozen
frozen_features.2.bn2.running_var        	--->	frozen
frozen_features.3.conv_pw.weight         	--->	frozen
frozen_features.3.bn1.weight             	--->	frozen
frozen_features.3.bn1.bias               	--->	frozen
frozen_features.3.bn1.runnin

2025-09-03 22:26:36.188888614 [I:onnxruntime:Default, constant_sharing.cc:248 ApplyImpl] Total shared scalar initializer count: 68
2025-09-03 22:26:36.188908398 [I:onnxruntime:Default, graph_transformer.cc:15 Apply] GraphTransformer ConstantSharing modified: 1 with status: OK
2025-09-03 22:26:36.189494737 [I:onnxruntime:Default, graph_transformer.cc:15 Apply] GraphTransformer LayerNormFusionL1 modified: 0 with status: OK
2025-09-03 22:26:36.189615360 [I:onnxruntime:Default, graph_transformer.cc:15 Apply] GraphTransformer CommonSubexpressionElimination modified: 0 with status: OK
2025-09-03 22:26:36.189646504 [I:onnxruntime:Default, graph_transformer.cc:15 Apply] GraphTransformer GeluFusionL1 modified: 0 with status: OK
2025-09-03 22:26:36.189664757 [I:onnxruntime:Default, graph_transformer.cc:15 Apply] GraphTransformer SimplifiedLayerNormFusion modified: 0 with status: OK
2025-09-03 22:26:36.189683588 [I:onnxruntime:Default, graph_transformer.cc:15 Apply] GraphTransformer FastGeluFusio

In [5]:
# Saving all the files to use them later for the training.
onnxblock.save_checkpoint(training_block.parameters(), artifacts_path / 'checkpoint')

ir_version = 9
opset_import_version = 17

training_model.ir_version = ir_version
# print('Simplifying ONNX training model...')
# training_model, check = simplify(training_model)
# assert check, "Simplified ONNX training model could not be validated"artifacts_optim_2_comp_1
onnx.save(training_model, artifacts_path / 'training_model.onnx')

optimizer_model.ir_version = ir_version
optimizer_model.opset_import[1].version = opset_import_version
onnx.save(optimizer_model, artifacts_path / 'optimizer_model.onnx')

eval_model.ir_version = ir_version
# print('Simplifying ONNX eval model...')
# eval_model, check = simplify(eval_model)
# assert check, "Simplified ONNX eval model could not be validated"
onnx.save(eval_model, artifacts_path / 'eval_model.onnx')

print(f'Artifacts saved in {artifacts_path} directory')

Artifacts saved in ../artifacts/mobilenetv2_100/frozen_layer_21_classes_10/cpu directory


## Verify model

In [6]:
from pathlib import Path

import numpy as np
from onnxruntime.training.api import CheckpointState, Module, Optimizer

In [8]:
# artifacts path
device = 'hailo'
artifacts_path = Path(f'../artifacts/{model_name}/frozen_layer_21_classes_10/hailo8_optim_3_comp_0')

print('Loading artifacts...')
# Create checkpoint state
state = CheckpointState.load_checkpoint(artifacts_path / 'checkpoint')

# Create module
print(f'Creating model with {device} device...')
model = Module(
    artifacts_path / 'training_model.onnx',
    state,
    artifacts_path / 'eval_model.onnx',
    device=device,
)

# Create optimizer
optimizer = Optimizer(artifacts_path / 'optimizer_model.onnx', model)
optimizer.set_learning_rate(0.001)

Loading artifacts...
Creating model with hailo device...


2025-09-03 22:26:49.508158386 [W:onnxruntime:, session_state.cc:1280 VerifyEachNodeIsAssignedToAnEp] Some nodes were not assigned to the preferred execution providers which may or may not have an negative impact on performance. e.g. ORT explicitly assigns shape related ops to CPU to improve perf.
2025-09-03 22:26:49.508174069 [W:onnxruntime:, session_state.cc:1282 VerifyEachNodeIsAssignedToAnEp] Rerunning with verbose output on a non-minimal build will show node assignments.
2025-09-03 22:26:49.536538968 [W:onnxruntime:, session_state.cc:1280 VerifyEachNodeIsAssignedToAnEp] Some nodes were not assigned to the preferred execution providers which may or may not have an negative impact on performance. e.g. ORT explicitly assigns shape related ops to CPU to improve perf.
2025-09-03 22:26:49.536550102 [W:onnxruntime:, session_state.cc:1282 VerifyEachNodeIsAssignedToAnEp] Rerunning with verbose output on a non-minimal build will show node assignments.


In [9]:
print(f'Model inputs: {model.input_names()}')
print(f'Model outputs: {model.output_names()}')

batch_size = 1
num_classes = 10  # Set this to your actual number of classes

# Prepare dummy input and label
forward_inputs = [
    np.ones((batch_size, 3, 224, 224), dtype=np.float32),  # input tensor
    np.zeros((batch_size,), dtype=np.int64),               # labels tensor
]

# If your model expects more inputs (check model.input_names()), add more dummy arrays as needed

model.train()
train_loss, logits = model(*forward_inputs)

Model inputs: ['input0', 'labels']
Model outputs: ['onnx::loss::32', 'post_output']
